# Supervised Learning Course Project

| Name | ID |
|------|----|
|  Mohamed Beshr Al Sofi | 20230717  |
|  Essam Mamdouh Saad    | 20230626  |
|  Eman Emad Farghaly    | 20230619  |
|  Malak Mohamed Saad    | 20230417  |

# Sentiment140 — Twitter Sentiment Analysis

**Dataset:** [Sentiment140 — 1.6 Million Tweets](https://www.kaggle.com/datasets/kazanova/sentiment140)

---

## Overview

This notebook walks through a complete pipeline for analysing sentiment from tweets using the **Sentiment140** dataset, which contains **1.6 million** tweets automatically labelled as positive or negative based on the presence of emoticons.

| Label | Meaning |
|-------|---------|
| 0     | Negative sentiment |
| 4     | Positive sentiment |



> **How to get the data:** Download `training.1600000.processed.noemoticon.csv` from the Kaggle link above and place it in the same directory as this notebook (or update the path in Section 2).

---
## 1. Setup and Imports

In [ ]:
# Core
import re
import string
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np
import pandas as pd

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

# Global plot style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.dpi':      120,
    'axes.titlesize':  14,
    'axes.labelsize':  12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('All libraries loaded successfully. ')

All libraries loaded successfully. 


---
## 2. Load the Dataset

The raw CSV has **no header row**. The six columns, in order, are:

| # | Column    | Description |
|---|-----------|-------------|
| 0 | `target`  | Polarity: **0** = negative, **4** = positive |
| 1 | `id`      | Tweet ID |
| 2 | `date`    | Date of tweet |
| 3 | `flag`    | Query keyword (NO_QUERY if absent) |
| 4 | `user`    | Twitter username |
| 5 | `text`    | Tweet content |

In [ ]:
from pathlib import Path

# Local copy (e.g. repo / notebook dir) or Kaggle dataset mount
_CANDIDATES = [
    Path('training.1600000.processed.noemoticon.csv'),
    Path('Sentiment-Fake-News-Analysis/training.1600000.processed.noemoticon.csv'),
    Path('/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv'),
]
DATA_PATH = None
for _p in _CANDIDATES:
    if _p.exists():
        DATA_PATH = str(_p.resolve())
        break
if DATA_PATH is None:
    raise FileNotFoundError(
        'training.1600000.processed.noemoticon.csv not found. '
        'Place it next to this notebook or set DATA_PATH manually.'
    )
print('Using data file:', DATA_PATH)

COLUMN_NAMES = ['target', 'id', 'date', 'flag', 'user', 'text']

df = pd.read_csv(DATA_PATH, encoding='latin-1', names=COLUMN_NAMES)

print('Shape:', df.shape)
df.head(5)


---
## 3. Data Cleaning

### 3.1 Initial Inspection

In [ ]:
# --- Data types ---
print('=== Data Types ===')
for col in df.columns:
    print(f'  {col:<15} {df[col].dtype}')

# --- Missing values (manual count) ---
print('\n=== Missing Values ===')
for col in df.columns:
    missing = 0
    for val in df[col]:
        if val != val:   # NaN check without isna()
            missing += 1
    print(f'  {col:<15} {missing}')

# --- Duplicate rows (manual count) ---
print('\n=== Duplicate Rows ===')
seen_texts = {}
dup_count  = 0
for tweet in df['text']:
    if tweet in seen_texts:
        dup_count += 1
    else:
        seen_texts[tweet] = True
print(f'  Total duplicates: {dup_count}')

# --- Target distribution (manual count) ---
print('\n=== Target Value Counts ===')
target_counts = {}
for val in df['target']:
    if val in target_counts:
        target_counts[val] += 1
    else:
        target_counts[val] = 1
for k, v in target_counts.items():
    print(f'  {k}: {v:,}')

### 3.2 Remap Target Labels and Parse Date

In [ ]:
# Remap: 0 -> Negative (0), 4 -> Positive (1)
sentiment_map = {0: 0, 4: 1}
label_map     = {0: 'Negative', 1: 'Positive'}

df['sentiment']       = df['target'].map(sentiment_map)
df['sentiment_label'] = df['sentiment'].map(label_map)

# Parse date
df['date']  = pd.to_datetime(df['date'], format='%a %b %d %H:%M:%S PDT %Y', errors='coerce')
df['month'] = df['date'].dt.month
df['hour']  = df['date'].dt.hour

# Drop unused columns
df.drop(columns=['id', 'flag', 'target'], inplace=True)

# Remove duplicate tweets
before = 0
for _ in df['text']:
    before += 1

df.drop_duplicates(subset='text', inplace=True)
df.dropna(subset=['text'], inplace=True)
df.reset_index(drop=True, inplace=True)

after = 0
for _ in df['text']:
    after += 1

print(f'Rows before cleaning : {before:,}')
print(f'Rows after  cleaning : {after:,}  (removed {before - after:,})')
df.head(3)

### 3.3 Text Preprocessing

Multi-step cleaning pipeline applied to every tweet:

| Step | Action |
|------|--------|
| 1 | Lowercase conversion |
| 2 | Remove URLs |
| 3 | Remove @mentions |
| 4 | Remove #hashtags |
| 5 | Remove HTML entities |
| 6 | Remove non-alphabetic characters and extra spaces |
| 7 | Remove English stopwords |
| 8 | Porter Stemming |

In [ ]:
stemmer        = PorterStemmer()
STOPWORD_LIST  = stopwords.words('english')
STOPWORD_DICT  = {}                          # use dict for O(1) lookup instead of 'in list'
for word in STOPWORD_LIST:
    STOPWORD_DICT[word] = True


def clean_tweet(text):
    """
    Full preprocessing pipeline for a single tweet.
    Avoids built-in convenience functions where possible.
    """
    # Step 1: lowercase
    result = ''
    for ch in text:
        if 'A' <= ch <= 'Z':
            result += chr(ord(ch) + 32)
        else:
            result += ch
    text = result

    # Steps 2-6: regex-based noise removal (regex is necessary for pattern matching)
    text = re.sub(r'http\S+|www\S+', ' ', text)   # URLs
    text = re.sub(r'@\w+',           ' ', text)   # @mentions
    text = re.sub(r'#\w+',           ' ', text)   # #hashtags
    text = re.sub(r'&[a-z]+;',       ' ', text)   # HTML entities
    text = re.sub(r'[^a-z ]',        ' ', text)   # non-alpha

    # Collapse multiple spaces manually
    compressed = ''
    prev_space = False
    for ch in text:
        if ch == ' ':
            if not prev_space:
                compressed += ch
            prev_space = True
        else:
            compressed += ch
            prev_space = False
    text = compressed.strip()

    # Step 7-8: tokenise, filter stopwords, stem
    tokens = word_tokenize(text)
    clean_tokens = []
    for token in tokens:
        # skip stopwords and very short tokens
        token_len = 0
        for _ in token:
            token_len += 1
        if token not in STOPWORD_DICT and token_len > 1:
            clean_tokens.append(stemmer.stem(token))

    # Rejoin manually
    joined = ''
    for i in range(0, len(clean_tokens)):
        if i > 0:
            joined += ' '
        joined += clean_tokens[i]
    return joined


print('Cleaning tweets... (may take several minutes on the full 1.6 M dataset)')
df['clean_text'] = df['text'].apply(clean_tweet)

# Tweet length features (manual character / word counts)
tweet_lengths     = []
word_counts       = []
clean_word_counts = []

for raw, cleaned in zip(df['text'], df['clean_text']):
    # character count
    char_c = 0
    for _ in raw:
        char_c += 1
    tweet_lengths.append(char_c)

    # raw word count
    wc = 0
    in_word = False
    for ch in raw:
        if ch != ' ':
            if not in_word:
                wc += 1
                in_word = True
        else:
            in_word = False
    word_counts.append(wc)

    # clean word count
    cwc = 0
    in_word = False
    for ch in cleaned:
        if ch != ' ':
            if not in_word:
                cwc += 1
                in_word = True
        else:
            in_word = False
    clean_word_counts.append(cwc)

df['tweet_length']     = tweet_lengths
df['word_count']       = word_counts
df['clean_word_count'] = clean_word_counts

print('Text preprocessing complete.')
df[['text', 'clean_text', 'tweet_length', 'word_count']].head(3)

---
## 4. Exploratory Data Analysis (EDA)

### 4.1 Dataset Summary Statistics

In [ ]:
# Manual summary statistics without .describe()

def manual_stats(values):
    """Compute count, mean, std, min, max, median manually."""
    n     = 0
    total = 0
    mn    = values[0]
    mx    = values[0]

    for v in values:
        n     += 1
        total += v
        if v < mn:
            mn = v
        if v > mx:
            mx = v

    mean   = total / n
    sq_sum = 0
    for v in values:
        sq_sum += (v - mean) ** 2
    std = (sq_sum / n) ** 0.5

    sorted_vals = sorted(values)
    mid = n // 2
    median = sorted_vals[mid] if n % 2 == 1 else (sorted_vals[mid - 1] + sorted_vals[mid]) / 2

    return {'count': n, 'mean': round(mean, 2), 'std': round(std, 2),
            'min': mn, 'median': median, 'max': mx}


# Count sentiments manually
neg_count = 0
pos_count = 0
for s in df['sentiment']:
    if s == 0:
        neg_count += 1
    else:
        pos_count += 1

total_count = neg_count + pos_count

print('=== General Statistics ===')
print(f'  Total tweets        : {total_count:,}')
print(f'  Negative tweets (0) : {neg_count:,}')
print(f'  Positive tweets (1) : {pos_count:,}')

print('\n=== Tweet Length Stats ===')
for col_name, col_data in [
    ('tweet_length',     df['tweet_length'].tolist()),
    ('word_count',       df['word_count'].tolist()),
    ('clean_word_count', df['clean_word_count'].tolist()),
]:
    s = manual_stats(col_data)
    print(f'  {col_name}:')
    for k, v in s.items():
        print(f'    {k:<8}: {v}')

### 4.2 Top Words per Sentiment

In [ ]:
def count_words(texts, top_n=20):
    """
    Build a word frequency dictionary manually and
    return the top_n entries sorted by count descending.
    """
    freq = {}
    for text in texts:
        # manual split on spaces
        word = ''
        for ch in text:
            if ch == ' ':
                if word:
                    if word in freq:
                        freq[word] += 1
                    else:
                        freq[word] = 1
                    word = ''
            else:
                word += ch
        if word:
            if word in freq:
                freq[word] += 1
            else:
                freq[word] = 1

    # manual sort (selection-sort style on items)
    items = []
    for k, v in freq.items():
        items.append((k, v))

    # selection-sort descending by count
    for i in range(0, top_n if top_n < len(items) else len(items)):
        max_idx = i
        for j in range(i + 1, len(items)):
            if items[j][1] > items[max_idx][1]:
                max_idx = j
        items[i], items[max_idx] = items[max_idx], items[i]

    return items[:top_n]


neg_texts = []
pos_texts = []
for text, sentiment in zip(df['clean_text'], df['sentiment']):
    if sentiment == 0:
        neg_texts.append(text)
    else:
        pos_texts.append(text)

neg_words = count_words(neg_texts, top_n=20)
pos_words = count_words(pos_texts, top_n=20)

print('Top 10 words — NEGATIVE tweets:')
for word, count in neg_words[:10]:
    print(f'  {word:<25} {count:,}')

print()
print('Top 10 words — POSITIVE tweets:')
for word, count in pos_words[:10]:
    print(f'  {word:<25} {count:,}')

---
## 5. Visualizations

### 5.1 Sentiment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette   = {'Negative': '#e74c3c', 'Positive': '#2ecc71'}

labels = ['Negative', 'Positive']
counts = [neg_count, pos_count]
colors = [palette['Negative'], palette['Positive']]

# Bar chart
bars = axes[0].bar(labels, counts, color=colors, edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars, counts):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5000,
        f'{val:,}', ha='center', va='bottom', fontweight='bold'
    )
axes[0].set_title('Sentiment Distribution (Bar)', fontweight='bold')
axes[0].set_ylabel('Number of Tweets')
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}')
)

# Pie chart
axes[1].pie(
    counts, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12}
)
axes[1].set_title('Sentiment Distribution (Pie)', fontweight='bold')

plt.suptitle('Overall Sentiment Distribution — Sentiment140',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_01_sentiment_distribution.png', bbox_inches='tight')
plt.show()

### 5.2 Tweet Length Distribution by Sentiment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

neg_lengths    = []
pos_lengths    = []
neg_word_count = []
pos_word_count = []

for tlen, wc, s in zip(df['tweet_length'], df['word_count'], df['sentiment']):
    if s == 0:
        neg_lengths.append(tlen)
        neg_word_count.append(wc)
    else:
        pos_lengths.append(tlen)
        pos_word_count.append(wc)

# Character length histogram
axes[0].hist(neg_lengths, bins=50, alpha=0.7,
             color=palette['Negative'], label='Negative', edgecolor='none')
axes[0].hist(pos_lengths, bins=50, alpha=0.7,
             color=palette['Positive'], label='Positive', edgecolor='none')
axes[0].set_title('Tweet Character Length by Sentiment', fontweight='bold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Word count box plot
bp = axes[1].boxplot(
    [neg_word_count, pos_word_count],
    labels=['Negative', 'Positive'],
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=2)
)
for patch, color in zip(bp['boxes'], [palette['Negative'], palette['Positive']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_title('Word Count Distribution by Sentiment', fontweight='bold')
axes[1].set_ylabel('Word Count')

plt.suptitle('Tweet Length Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_02_tweet_length.png', bbox_inches='tight')
plt.show()

### 5.3 Top 20 Words per Sentiment (Bar Charts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, word_list, label, color in [
    (axes[0], neg_words, 'Negative', '#e74c3c'),
    (axes[1], pos_words, 'Positive', '#2ecc71'),
]:
    # Reverse for horizontal bar (highest at top)
    words_rev  = [w for w, _ in word_list[::-1]]
    counts_rev = [c for _, c in word_list[::-1]]

    bars = ax.barh(words_rev, counts_rev, color=color, alpha=0.85, edgecolor='white')
    max_c = 0
    for c in counts_rev:
        if c > max_c:
            max_c = c

    for bar, val in zip(bars, counts_rev):
        ax.text(
            bar.get_width() + max_c * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=9
        )

    ax.set_title(f'Top 20 Words — {label} Tweets', fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{int(x):,}')
    )

plt.suptitle('Most Frequent Words After Cleaning',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_03_top_words.png', bbox_inches='tight')
plt.show()

### 5.4 Tweets by Hour of Day

In [ ]:
# Build hour frequency tables manually
neg_hourly = {}
pos_hourly = {}
for h in range(0, 24):
    neg_hourly[h] = 0
    pos_hourly[h] = 0

for h, s in zip(df['hour'], df['sentiment']):
    if h != h:   # NaN guard
        continue
    hour_int = int(h)
    if s == 0:
        neg_hourly[hour_int] += 1
    else:
        pos_hourly[hour_int] += 1

hours = list(range(0, 24))
neg_hourly_vals = [neg_hourly[h] for h in hours]
pos_hourly_vals = [pos_hourly[h] for h in hours]

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(hours, neg_hourly_vals, marker='o', linewidth=2.5, markersize=5,
        color=palette['Negative'], label='Negative', alpha=0.9)
ax.plot(hours, pos_hourly_vals, marker='o', linewidth=2.5, markersize=5,
        color=palette['Positive'], label='Positive', alpha=0.9)

ax.set_title('Tweets per Hour of Day by Sentiment (PDT)', fontweight='bold')
ax.set_xlabel('Hour of Day (24-h)')
ax.set_ylabel('Number of Tweets')
ax.set_xticks(hours)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()

plt.tight_layout()
plt.savefig('plot_05_tweets_by_hour.png', bbox_inches='tight')
plt.show()

### 5.5 Tweets by Month

In [ ]:
import calendar

# Build month frequency tables manually
months = list(range(1, 13))
neg_monthly = {}
pos_monthly = {}
for m in months:
    neg_monthly[m] = 0
    pos_monthly[m] = 0

for m, s in zip(df['month'], df['sentiment']):
    if m != m:
        continue
    m_int = int(m)
    if s == 0:
        neg_monthly[m_int] += 1
    else:
        pos_monthly[m_int] += 1

month_labels = [calendar.month_abbr[m] for m in months]
neg_m_vals   = [neg_monthly[m] for m in months]
pos_m_vals   = [pos_monthly[m] for m in months]

# Position bars side by side
bar_width = 0.35
x     = [i for i in range(0, len(months))]
x_neg = [xi - bar_width / 2 for xi in x]
x_pos = [xi + bar_width / 2 for xi in x]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x_neg, neg_m_vals, width=bar_width,
       color=palette['Negative'], edgecolor='white', label='Negative')
ax.bar(x_pos, pos_m_vals, width=bar_width,
       color=palette['Positive'], edgecolor='white', label='Positive')

ax.set_xticks(x)
ax.set_xticklabels(month_labels)
ax.set_title('Tweets by Month and Sentiment', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Tweets')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(title='Sentiment')

plt.tight_layout()
plt.savefig('plot_06_tweets_by_month.png', bbox_inches='tight')
plt.show()

### 5.6 Clean Word Count Distribution (KDE)

In [ ]:
neg_cwc = []
pos_cwc = []
for cwc, s in zip(df['clean_word_count'], df['sentiment']):
    if s == 0:
        neg_cwc.append(cwc)
    else:
        pos_cwc.append(cwc)

fig, ax = plt.subplots(figsize=(12, 5))

sns.kdeplot(neg_cwc, ax=ax, fill=True, alpha=0.4,
            color=palette['Negative'], label='Negative', linewidth=2)
sns.kdeplot(pos_cwc, ax=ax, fill=True, alpha=0.4,
            color=palette['Positive'], label='Positive', linewidth=2)

ax.set_title('KDE — Clean Word Count per Tweet by Sentiment', fontweight='bold')
ax.set_xlabel('Number of Words (after cleaning)')
ax.set_ylabel('Density')
ax.legend()
ax.set_xlim(0, 40)

plt.tight_layout()
plt.savefig('plot_07_kde_word_count.png', bbox_inches='tight')
plt.show()

### 5.7 Correlation Heatmap (Numeric Features)

In [ ]:
numeric_cols = ['sentiment', 'tweet_length', 'word_count', 'clean_word_count', 'hour', 'month']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.3f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    square=True, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Heatmap — Numeric Features', fontweight='bold')

plt.tight_layout()
plt.savefig('plot_08_correlation_heatmap.png', bbox_inches='tight')
plt.show()

### 5.8 Top Users by Tweet Count

In [ ]:
# Build user frequency dict manually
user_freq = {}
for user in df['user']:
    if user in user_freq:
        user_freq[user] += 1
    else:
        user_freq[user] = 1

# Manual selection sort to extract top 15
user_items = [[k, v] for k, v in user_freq.items()]
for i in range(0, 15):
    max_idx = i
    for j in range(i + 1, len(user_items)):
        if user_items[j][1] > user_items[max_idx][1]:
            max_idx = j
    user_items[i], user_items[max_idx] = user_items[max_idx], user_items[i]

top_users   = [row[0] for row in user_items[:15]]
top_ucounts = [row[1] for row in user_items[:15]]

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = plt.cm.plasma(np.linspace(0.2, 0.8, 15))
bars = ax.bar(top_users, top_ucounts, color=bar_colors, edgecolor='white')

max_uc = 0
for c in top_ucounts:
    if c > max_uc:
        max_uc = c

for bar, val in zip(bars, top_ucounts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_uc * 0.01,
        str(val), ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

ax.set_title('Top 15 Most Active Users', fontweight='bold')
ax.set_xlabel('Username')
ax.set_ylabel('Number of Tweets')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('plot_09_top_users.png', bbox_inches='tight')
plt.show()

---
## 6. TF-IDF

In [ ]:
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(df['clean_text'])
Y = df['sentiment'].values



---
## 7. KNN + Naïve Bayes Classification

**Person 3** — KNN + Naïve Bayes models

This section trains two classical text classifiers on the TF-IDF features built above:

| Model | Why it suits text |
|-------|-------------------|
| **K-Nearest Neighbours (KNN)** | Cosine similarity between TF-IDF vectors places semantically close tweets together |
| **Multinomial Naïve Bayes** | Natural fit for bag-of-words / TF-IDF: models word-count likelihoods per class |

### 7.1 Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    tfidf_matrix, Y,
    test_size=0.20,
    random_state=42,
    stratify=Y          # keep class balance in both splits
)

print('Train size:', X_train.shape[0])
print('Test  size:', X_test.shape[0])

### 7.2 K-Nearest Neighbours (KNN)

#### 7.2.0 Dimensionality Reduction with TruncatedSVD (LSA)

The full TF-IDF matrix has tens of thousands of features, which makes KNN very slow
(it computes distances to every training point at prediction time).
**TruncatedSVD** (Latent Semantic Analysis) compresses the sparse TF-IDF vectors down to
a dense low-dimensional representation **before KNN** — drastically cutting distance-computation time
while preserving the most important variance in the data.

> This reduction is applied **only for KNN**. The original  /  (full TF-IDF) are kept
> unchanged for Naïve Bayes and all other cells.


In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# so slicing rows does not reduce the memory SVD needs.
SAMPLE_PER_CLASS = 20_000

# Get the original text + label arrays aligned with the train split
train_idx = np.array(y_train.index if hasattr(y_train, 'index') else range(len(y_train)))

# Use df directly 
neg_rows = df[df['sentiment'] == 0].sample(n=SAMPLE_PER_CLASS, random_state=42)
pos_rows = df[df['sentiment'] == 1].sample(n=SAMPLE_PER_CLASS, random_state=42)
df_knn   = pd.concat([neg_rows, pos_rows], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

texts_knn = df_knn['clean_text'].values
y_knn_all = df_knn['sentiment'].values

# Split this small sample into train/test
from sklearn.model_selection import train_test_split as tts
texts_tr, texts_te, y_train_knn, y_test_knn = tts(
    texts_knn, y_knn_all, test_size=0.2, random_state=42, stratify=y_knn_all
)

# Build a fresh TF-IDF on this small corpus only
vec_knn = TfidfVectorizer(ngram_range=(1, 2), max_features=30_000, sublinear_tf=True)
X_small_train = vec_knn.fit_transform(texts_tr)
X_small_test  = vec_knn.transform(texts_te)   

print(f'Small TF-IDF train : {X_small_train.shape}')
print(f'Small TF-IDF test  : {X_small_test.shape}')

# TruncatedSVD on the small matrix
N_COMPONENTS = 100
svd        = TruncatedSVD(n_components=N_COMPONENTS, random_state=42, algorithm='randomized')
normalizer = Normalizer(copy=False)

print(f'Running TruncatedSVD ({N_COMPONENTS} components)...')
X_train_knn = normalizer.fit_transform(svd.fit_transform(X_small_train))
X_test_knn  = normalizer.transform(svd.transform(X_small_test))

explained = svd.explained_variance_ratio_.sum()
print(f'Reduced train : {X_train_knn.shape}')
print(f'Reduced test  : {X_test_knn.shape}')
print(f'Variance kept : {explained*100:.1f}%')


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Train KNN ---
knn = KNeighborsClassifier(
    n_neighbors=5,
    metric='euclidean',
    algorithm='ball_tree',
    n_jobs=-1
)
knn.fit(X_train_knn, y_train_knn)

# --- Predict ---
y_pred_knn = knn.predict(X_test_knn)

# --- Metrics ---
knn_acc = accuracy_score(y_test_knn, y_pred_knn)
print(f'KNN Accuracy: {knn_acc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_knn, y_pred_knn, target_names=['Negative', 'Positive']))


In [ ]:
# --- Confusion Matrix — KNN ---
cm_knn = confusion_matrix(y_test_knn, y_pred_knn)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_knn, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive'],
    ax=ax
)
ax.set_title('KNN — Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()

#### 7.2.1 Choosing *k* — Elbow Plot

We sweep `k` from 1 to 19 (odd values only to avoid ties) and record test accuracy.

In [ ]:
k_values  = list(range(1, 20, 2))   # 1, 3, 5, ..., 19
k_acc     = []

for k in k_values:
    clf = KNeighborsClassifier(
        n_neighbors=k,
        metric='euclidean',
        algorithm='ball_tree',
        n_jobs=-1
    )
    clf.fit(X_train_knn, y_train_knn)
    acc = accuracy_score(y_test_knn, clf.predict(X_test_knn))
    k_acc.append(acc)
    print(f'  k={k:2d}  accuracy={acc:.4f}')

# --- Plot ---
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_values, k_acc, marker='o', color='#3498db', linewidth=2)
ax.set_xlabel('k  (number of neighbours)')
ax.set_ylabel('Test Accuracy')
ax.set_title('KNN — Accuracy vs k', fontsize=13, fontweight='bold')
ax.set_xticks(k_values)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

best_k   = k_values[int(np.argmax(k_acc))]
best_acc = max(k_acc)
print(f'\nBest k = {best_k}  (accuracy = {best_acc:.4f})')


### 7.3 Naïve Bayes Classifier

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MaxAbsScaler

# TF-IDF values can be negative after sublinear_tf, scale to [0, 1] for MultinomialNB
scaler   = MaxAbsScaler()
X_train_nb = scaler.fit_transform(X_train)
X_test_nb  = scaler.transform(X_test)

# --- Train Naïve Bayes ---
nb_clf = MultinomialNB(alpha=1.0)   
nb_clf.fit(X_train_nb, y_train)

# --- Predict ---
y_pred_nb = nb_clf.predict(X_test_nb)

# --- Metrics ---
nb_acc = accuracy_score(y_test, y_pred_nb)
print(f'Naïve Bayes Accuracy: {nb_acc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_nb, target_names=['Negative', 'Positive']))

In [ ]:
# --- Confusion Matrix — Naïve Bayes ---
cm_nb = confusion_matrix(y_test, y_pred_nb)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_nb, annot=True, fmt='d', cmap='Greens',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive'],
    ax=ax
)
ax.set_title('Naïve Bayes — Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()

#### 7.3.1 Smoothing Sensitivity — α Sweep

Laplace smoothing (`alpha`) controls how much probability mass is assigned to unseen n-grams.

In [ ]:
alphas   = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0]
nb_accs  = []

for a in alphas:
    clf = MultinomialNB(alpha=a)
    clf.fit(X_train_nb, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test_nb))
    nb_accs.append(acc)
    print(f'  alpha={a:<5}  accuracy={acc:.4f}')

# --- Plot ---
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(alphas, nb_accs, marker='s', color='#2ecc71', linewidth=2)
ax.set_xscale('log')
ax.set_xlabel('alpha  (Laplace smoothing)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Naïve Bayes — Accuracy vs alpha', fontsize=13, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

best_alpha = alphas[int(np.argmax(nb_accs))]
best_nb_acc = max(nb_accs)
print(f'\nBest alpha = {best_alpha}  (accuracy = {best_nb_acc:.4f})')

### 7.4 Model Comparison

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# KNN is evaluated against y_test_knn (8k sample)
# Naïve Bayes is evaluated against y_test (full test set)
results = {
    'KNN': {
        'Accuracy' : accuracy_score(y_test_knn, y_pred_knn),
        'Precision': precision_score(y_test_knn, y_pred_knn, average='weighted'),
        'Recall'   : recall_score(y_test_knn, y_pred_knn, average='weighted'),
        'F1-Score' : f1_score(y_test_knn, y_pred_knn, average='weighted'),
    },
    'Naïve Bayes': {
        'Accuracy' : accuracy_score(y_test, y_pred_nb),
        'Precision': precision_score(y_test, y_pred_nb, average='weighted'),
        'Recall'   : recall_score(y_test, y_pred_nb, average='weighted'),
        'F1-Score' : f1_score(y_test, y_pred_nb, average='weighted'),
    }
}

results_df = pd.DataFrame(results).T
print(results_df.to_string(float_format='{:.4f}'.format))

# --- Side-by-side bar chart ---
metrics = list(results_df.columns)
x       = np.arange(len(metrics))
width   = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, results_df.loc['KNN'],         width, label='KNN',         color='#3498db')
bars2 = ax.bar(x + width/2, results_df.loc['Naïve Bayes'], width, label='Naïve Bayes', color='#2ecc71')

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('KNN vs Naïve Bayes — Performance Comparison', fontsize=13, fontweight='bold')
ax.legend()

for bar in list(bars1) + list(bars2):
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


---
## 8. Ensemble Models

In [ ]:
# =========================================
#  Dimensionality Reduction (TruncatedSVD)
# =========================================

from sklearn.decomposition import TruncatedSVD
import numpy as np
import matplotlib.pyplot as plt

N_COMPONENTS = 50

svd_full = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
X_train_svd_full = svd_full.fit_transform(X_train)
X_test_svd_full  = svd_full.transform(X_test)

explained_var = svd_full.explained_variance_ratio_
cum_var = np.cumsum(explained_var)

print(f"Total Explained Variance ({N_COMPONENTS} comps): {cum_var[-1]*100:.2f}%")

# Plot Explained Variance
plt.figure(figsize=(8,5))
plt.plot(cum_var, linewidth=2)
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("TruncatedSVD - Explained Variance")
plt.grid(True)
plt.show()

### 8.1 Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns

rf = RandomForestClassifier(
    n_estimators=30,
    max_depth=15,
    random_state=42,
    max_features='sqrt',
    n_jobs=-1
)

rf.fit(X_train_svd_full, y_train)
y_pred_rf = rf.predict(X_test_svd_full)

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1  = f1_score(y_test, y_pred_rf, average='weighted')

print(f"Random Forest Accuracy: {rf_acc:.4f}")
print(classification_report(y_test, y_pred_rf))


cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6,5))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'])
plt.title("Random Forest - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

SAMPLE_SIZE = 100000

idx = np.random.choice(len(X_train_svd_full), SAMPLE_SIZE, replace=False)
X_sample = X_train_svd_full[idx]
y_sample = y_train[idx]

param_dist = {
    'n_estimators': [30, 50, 70],
    'max_depth': [10, 15, 20],
}

random_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=2,       
    cv=2,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_rf.fit(X_sample, y_sample)

print("Best Parameters:", random_rf.best_params_)

best_rf = RandomForestClassifier(
    **random_rf.best_params_,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train_svd_full, y_train)
y_pred_rf_tuned = best_rf.predict(X_test_svd_full)

rf_tuned_acc = accuracy_score(y_test, y_pred_rf_tuned)
rf_tuned_f1  = f1_score(y_test, y_pred_rf_tuned, average='weighted')

print(f"Tuned RF Accuracy: {rf_tuned_acc:.4f}")

cm_xgb = confusion_matrix(y_test, y_pred_rf_tuned)

plt.figure(figsize=(6,5))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'])

plt.title("Tuned RF - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### 8.2 XGBoost Classifier 

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train_svd_full, y_train)
y_pred_xgb = xgb.predict(X_test_svd_full)

xgb_acc = accuracy_score(y_test, y_pred_xgb)
xgb_f1  = f1_score(y_test, y_pred_xgb, average='weighted')

print(f"XGBoost Accuracy: {xgb_acc:.4f}")
print(classification_report(y_test, y_pred_xgb))


cm_xgb = confusion_matrix(y_test, y_pred_xgb)

plt.figure(figsize=(6,5))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'])
plt.title("XGBoost - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
SAMPLE_SIZE = 100000
idx = np.random.choice(len(X_train_svd_full), SAMPLE_SIZE, replace=False)

X_sample = X_train_svd_full[idx]
y_sample = y_train[idx]

param_dist_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0]
}

random_xgb = RandomizedSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    param_distributions=param_dist_xgb,
    n_iter=8,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_xgb.fit(X_sample, y_sample)

print("Best XGB Params:", random_xgb.best_params_)

best_xgb = XGBClassifier(
    **random_xgb.best_params_,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

best_xgb.fit(X_train_svd_full, y_train)

y_pred_xgb_tuned = best_xgb.predict(X_test_svd_full)

xgb_acc_tuned = accuracy_score(y_test, y_pred_xgb_tuned)
xgb_f1_tuned  = f1_score(y_test, y_pred_xgb_tuned, average='weighted')

print(f"Tuned XGBoost Accuracy: {xgb_acc_tuned:.4f}")

cm_xgb = confusion_matrix(y_test, y_pred_xgb_tuned)

plt.figure(figsize=(6,5))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'])

plt.title("Tuned XGBoost - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

---
## 9. Final Model Comparison

In [ ]:
# =========================================
#  ROC Curve
# =========================================

from sklearn.metrics import roc_curve, auc

rf_probs  = best_rf.predict_proba(X_test_svd_full)[:,1]
xgb_probs = xgb.predict_proba(X_test_svd_full)[:,1]
nb_probs  = nb_clf.predict_proba(X_test_nb)[:,1]

# ROC
fpr_rf, tpr_rf, _   = roc_curve(y_test, rf_probs)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_probs)
fpr_nb, tpr_nb, _   = roc_curve(y_test, nb_probs)

auc_rf  = auc(fpr_rf, tpr_rf)
auc_xgb = auc(fpr_xgb, tpr_xgb)
auc_nb  = auc(fpr_nb, tpr_nb)

plt.figure(figsize=(8,6))
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={auc_rf:.3f})")
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC={auc_xgb:.3f})")
plt.plot(fpr_nb, tpr_nb, label=f"Naive Bayes (AUC={auc_nb:.3f})")

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score

results = {
    'KNN': {
        'Accuracy': accuracy_score(y_test_knn, y_pred_knn),
        'F1': f1_score(y_test_knn, y_pred_knn, average='weighted')
    },
    'Naive Bayes': {
        'Accuracy': accuracy_score(y_test, y_pred_nb),
        'F1': f1_score(y_test, y_pred_nb, average='weighted')
    },
    'Random Forest': {
        'Accuracy': rf_acc,
        'F1': rf_f1
    },
    'RF Tuned': {
       'Accuracy': rf_tuned_acc,
        'F1': rf_tuned_f1
    },
    'XGBoost': {
        'Accuracy': xgb_acc,
        'F1': xgb_f1
    },
    'XGB Tuned': {
        'Accuracy': xgb_acc_tuned ,
        'F1': xgb_f1_tuned
    }
}

results_df = pd.DataFrame(results).T
print(results_df)

results_df.sort_values(by='F1', ascending=False).plot(kind='bar', figsize=(10,5))
plt.title("Final Model Ranking")
plt.ylabel("Score")
plt.grid(axis='y')
plt.show()

## Tokenization and Padding

`clean_text` is already lowercased, stemmed, and space-separated. Here we **tokenize** by splitting on spaces (no extra NLTK calls), build a **vocabulary** with integer IDs (`<PAD>` = 0, `<UNK>` = 1), and **pad or truncate** every sequence to a fixed length using plain Python loops and `numpy` — not `keras.preprocessing` or `sklearn` vectorizers.


### Tokenization helpers

Constants and manual space-splitting; counting word frequencies over the corpus.

In [ ]:
# --- Constants ---
PAD_ID = 0
UNK_ID = 1


def tokenize_clean_text(text):
    """Split clean_text into tokens (manual space split, matches notebook style)."""
    if text is None:
        return []
    tokens = []
    current = ''
    for ch in text:
        if ch == ' ':
            if current:
                tokens.append(current)
                current = ''
        else:
            current += ch
    if current:
        tokens.append(current)
    return tokens


def count_word_frequencies(text_series):
    freq = {}
    for text in text_series:
        for tok in tokenize_clean_text(text):
            if tok in freq:
                freq[tok] += 1
            else:
                freq[tok] = 1
    return freq


### Vocabulary mapping

Reserve ids for `<PAD>` and `<UNK>`; assign remaining ids by frequency.

In [ ]:
def build_vocab_from_freq(freq, max_vocab=20000, min_freq=2):
    """Ids 0 = <PAD>, 1 = <UNK>; remaining ids by descending frequency."""
    items = []
    for w, c in freq.items():
        if c >= min_freq:
            items.append((w, c))
    items.sort(key=lambda x: (-x[1], x[0]))
    cap = max(0, max_vocab - 2)
    chosen = items[:cap]
    word_to_id = {'<PAD>': PAD_ID, '<UNK>': UNK_ID}
    id_to_word = {PAD_ID: '<PAD>', UNK_ID: '<UNK>'}
    next_id = 2
    for w, _ in chosen:
        word_to_id[w] = next_id
        id_to_word[next_id] = w
        next_id += 1
    return word_to_id, id_to_word


def encode_tokens(tokens, word_to_id):
    out = []
    unk = word_to_id.get('<UNK>', UNK_ID)
    for t in tokens:
        if t in word_to_id:
            out.append(word_to_id[t])
        else:
            out.append(unk)
    return out


### Padding

Post-pad or truncate each encoded sequence; stack rows into a `numpy` matrix.

In [ ]:
def pad_sequence_ids(ids, max_len, pad_value=PAD_ID):
    """Post-pad or truncate to length max_len."""
    n = len(ids)
    if n >= max_len:
        return ids[:max_len]
    out = []
    i = 0
    while i < max_len:
        if i < n:
            out.append(ids[i])
        else:
            out.append(pad_value)
        i += 1
    return out


def texts_to_padded_matrix(text_series, word_to_id, max_len):
    """Shape (n_samples, max_len), dtype int32."""
    n = len(text_series)
    X = np.zeros((n, max_len), dtype=np.int32)
    for row, text in enumerate(text_series):
        toks = tokenize_clean_text(text)
        ids = encode_tokens(toks, word_to_id)
        seq = pad_sequence_ids(ids, max_len, PAD_ID)
        X[row, :] = np.asarray(seq, dtype=np.int32)
    return X


### Word frequencies and vocabulary

Scan all `clean_text` rows (can take a few minutes on ~1.6M tweets).

In [ ]:
print('Building word frequencies...')
word_freq = count_word_frequencies(df['clean_text'])

MAX_VOCAB = 20000
MIN_FREQ = 2
word_to_id, id_to_word = build_vocab_from_freq(word_freq, max_vocab=MAX_VOCAB, min_freq=MIN_FREQ)
VOCAB_SIZE = len(word_to_id)
print('Vocabulary size (incl. PAD & UNK):', VOCAB_SIZE)


### Sequence length and padded matrix

Set `SEQ_MAX_LEN` from the 95th percentile of token counts (clamped), then build `X_token_padded`.

In [ ]:
lengths = []
for text in df['clean_text']:
    lengths.append(len(tokenize_clean_text(text)))
lengths = np.array(lengths, dtype=np.int64)
SEQ_MAX_LEN = int(np.percentile(lengths, 95))
SEQ_MAX_LEN = max(SEQ_MAX_LEN, 8)
SEQ_MAX_LEN = min(SEQ_MAX_LEN, 128)
print(f'Token length 95th pct: {np.percentile(lengths, 95):.1f}  -> SEQ_MAX_LEN = {SEQ_MAX_LEN}')

print('Building padded integer matrix X_token_padded...')
X_token_padded = texts_to_padded_matrix(df['clean_text'], word_to_id, SEQ_MAX_LEN)
print('X_token_padded shape:', X_token_padded.shape, 'dtype:', X_token_padded.dtype)
print('First row (ids):', X_token_padded[0][:20], '...')


## Embedding Words

In [ ]:
# Load the pre-trained GloVe word embeddings (200-dimensional vectors).
# Each line in the file has the format: <word> <v1> <v2> ... <v200>
# We store the result as a dict: { word (str) -> vector (np.ndarray) }
glove = {}
with open("/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.200d.txt",
          encoding="utf8") as f:
    for line in f:
        parts  = line.split()
        word   = parts[0]                                  # the word token
        vector = np.array(parts[1:], dtype=np.float32)    # 200-d embedding
        glove[word] = vector


In [ ]:
vocab_size    = len(word_to_id)   # total number of tokens in our vocabulary
embedding_dim = 200               # GloVe vector dimension (must match the file)

def wordToVectors(words_dict):
    """
    Build an embedding matrix of shape (vocab_size, embedding_dim).

    For each word in our vocabulary, look up its GloVe vector.
    Words not found in GloVe are left as zero vectors (random initialisation
    is implicitly handled by Keras when a row is all-zeros at training time).
    """
    embedding_matrix = np.zeros((vocab_size, embedding_dim))

    for word, idx in words_dict.items():
        vector = glove.get(word)        # None if the word is out-of-vocabulary
        if vector is not None:
            embedding_matrix[idx] = vector

    return embedding_matrix

# Build the matrix that will be fed into the Keras Embedding layer
embedding_matrix = wordToVectors(word_to_id)


## CNN Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

# Build a 1-D Convolutional Neural Network for binary sentiment classification.
model = Sequential([
    # --- Embedding layer ---
    # Uses the pre-trained GloVe weights; kept frozen (trainable=False) so the
    # model learns higher-level features without distorting the embeddings.
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=SEQ_MAX_LEN,
        trainable=False
    ),

    # --- Feature extraction ---
    # 1-D convolution scans the sequence with 128 filters of width 3 tokens,
    # learning local n-gram-like patterns relevant to sentiment.
    Conv1D(filters=128, kernel_size=3, activation='relu'),

    # Global max-pooling picks the strongest activation across the sequence,
    # compressing the feature map to a fixed-size vector regardless of length.
    GlobalMaxPooling1D(),

    # --- Classification head ---
    Dense(64, activation='relu'),   # intermediate fully-connected layer
    Dropout(0.5),                   # regularisation – prevents over-fitting

    # Output: single sigmoid unit → probability of positive sentiment
    Dense(1, activation='sigmoid')
])

# Binary cross-entropy is the standard loss for binary classification.
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
from sklearn.model_selection import train_test_split

# Split the padded token matrix and labels into three disjoint subsets:
#   60 % training   – used to update model weights
#   20 % validation – monitored during training to detect over-fitting
#   20 % test       – held out entirely; used only for final evaluation
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_token_padded, Y,
    test_size=0.40,     # remaining 40 % goes into a temporary pool
    random_state=42,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp,       # split the temporary pool 50/50 → 20 % each
    test_size=0.50,
    random_state=42,
)


In [ ]:
# Train the CNN for 10 epochs with mini-batches of 32 samples.
# Validation metrics are logged at the end of every epoch so we can
# spot over-fitting early (validation loss rising while training loss falls).
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

In [ ]:
# Run the trained model on the held-out test set.
# `pred` contains raw sigmoid probabilities in the range [0, 1].
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

pred = model.predict(X_test)
pred = (pred > 0.5).astype(int)

acc = accuracy_score(y_test, pred)
f1  = f1_score(y_test, pred)

print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred))

print("\nClassification Report:")
print(classification_report(y_test, pred))

In [ ]:

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train', 'Validation'])
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['Train', 'Validation'])
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
model2 = Sequential([
    Embedding(vocab_size, embedding_dim, weights=[embedding_matrix],
              input_length=SEQ_MAX_LEN, trainable=False),
    Conv1D(128, 3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3), 
    Dense(1, activation='sigmoid')
])

model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history2 = model2.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

pred2 = (model2.predict(X_test) > 0.5).astype(int)
acc2 = accuracy_score(y_test, pred2)
f12  = f1_score(y_test, pred2)

print(f"Model2 Accuracy: {acc2:.4f}")
print(f"Model2 F1: {f12:.4f}")

In [ ]:
model3 = Sequential([
    Embedding(vocab_size, embedding_dim, weights=[embedding_matrix],
              input_length=SEQ_MAX_LEN, trainable=True), 
    Conv1D(128, 3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history3 = model3.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

pred3 = (model3.predict(X_test) > 0.5).astype(int)
acc3 = accuracy_score(y_test, pred3)
f13  = f1_score(y_test, pred3)

print(f"Model3 Accuracy: {acc3:.4f}")
print(f"Model3 F1: {f13:.4f}")

In [ ]:
print("\n=== Model Comparison ===")
print(f"Baseline  -> Acc: {acc:.4f}, F1: {f1:.4f}")
print(f"Dropout   -> Acc: {acc2:.4f}, F1: {f12:.4f}")
print(f"Trainable -> Acc: {acc3:.4f}, F1: {f13:.4f}")

## Autoencoder


In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, RepeatVector
from tensorflow.keras.callbacks import EarlyStopping


X_sample = X_token_padded.astype("float32") / vocab_size

X_train_ae, X_test_ae = train_test_split(
    X_sample,
    test_size=0.2,
    random_state=42
)


# --------------------------------
#  Build Autoencoder
# --------------------------------
input_seq = Input(shape=(SEQ_MAX_LEN,))

# Embedding layer
embedding = Embedding(
    input_dim=vocab_size,
    output_dim=64,          
    input_length=SEQ_MAX_LEN,
    trainable=False
)(input_seq)

# Encoder
encoded = GlobalAveragePooling1D()(embedding)
encoded = Dense(32, activation='relu')(encoded)

# Decoder
decoded = RepeatVector(SEQ_MAX_LEN)(encoded)
decoded = Dense(1, activation='sigmoid')(decoded)

# Model
autoencoder = Model(inputs=input_seq, outputs=decoded)

autoencoder.compile(
    optimizer='adam',
    loss='mse'
)

autoencoder.summary()

In [ ]:
early_stop = EarlyStopping(
    patience=3,
    restore_best_weights=True
)

history_ae = autoencoder.fit(
    X_train_ae,
    np.expand_dims(X_train_ae, axis=-1),
    epochs=10,
    batch_size=64,
    validation_data=(X_test_ae, np.expand_dims(X_test_ae, axis=-1)),
    callbacks=[early_stop]
)

In [ ]:
plt.plot(history_ae.history['loss'])
plt.plot(history_ae.history['val_loss'])
plt.legend(['Train', 'Validation'])
plt.title('Autoencoder Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()


In [ ]:
reconstructed = autoencoder.predict(X_test_ae)

mse = np.mean(
    (np.expand_dims(X_test_ae, axis=-1) - reconstructed) ** 2,
    axis=(1, 2)
)

print("Sample reconstruction errors:", mse[:10])

In [ ]:
threshold = np.percentile(mse, 95)
print("Threshold:", threshold)

In [ ]:
anomalies = mse > threshold

print("Number of anomalies:", np.sum(anomalies))

In [ ]:
anom_idx = np.where(anomalies == True)[0]

for i in np.random.choice(anom_idx, 5, replace=False):
    print("Anomalous text:")
    print(df['clean_text'].iloc[i])
    print("Error:", mse[i])
    print("------")

In [ ]:
plt.hist(mse, bins=50)
plt.axvline(threshold)
plt.title("Reconstruction Error Distribution")
plt.show()

---
## 9b. Additional CNN Experiments — Base Model (No Embeddings) & Optimizer Comparison

> Train a baseline CNN with **no pre-trained embeddings** to measure the raw gain from GloVe/FastText, then experiment with **RMSprop** and **SGD+Momentum** optimizers to find the best training configuration. Results feed directly into the final comparison section.


### 9b.1 Shared Train / Val / Test Split for Experiments

All experiments below use the same fixed split so results are directly comparable.

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

Y_dl = df['sentiment'].values.astype('float32')

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_token_padded, Y_dl, test_size=0.40, random_state=42)
X_vl, X_te, y_vl, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42)

print(f'Train: {X_tr.shape[0]:,}  |  Val: {X_vl.shape[0]:,}  |  Test: {X_te.shape[0]:,}')


### 9b.2 Base Model — No Pre-trained Embeddings (Random Init)

This model uses the **same CNN architecture** but initialises the embedding layer **randomly** (no GloVe weights).  
Comparing it against `model` (GloVe frozen) shows exactly how much value the pre-trained embeddings add.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Base model: random embeddings, trainable, everything else identical
model_no_emb = Sequential([
    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,      # same dimension as GloVe for fair comparison
              input_length=SEQ_MAX_LEN,
              trainable=True),               # learns from scratch no pretrained weights
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid'),
], name='CNN_No_Pretrained_Embeddings')

model_no_emb.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model_no_emb.summary()


In [ ]:
es_no_emb = EarlyStopping(monitor='val_loss', patience=3,
                           restore_best_weights=True, verbose=1)

history_no_emb = model_no_emb.fit(
    X_tr, y_tr,
    epochs=10, batch_size=256,
    validation_data=(X_vl, y_vl),
    callbacks=[es_no_emb],
    verbose=1,
)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

proba_no_emb = model_no_emb.predict(X_te, verbose=0).ravel()
pred_no_emb  = (proba_no_emb > 0.5).astype(int)

acc_no_emb = accuracy_score(y_te, pred_no_emb)
f1_no_emb  = f1_score(y_te, pred_no_emb)

print(f'CNN (no pretrained emb) — Accuracy: {acc_no_emb:.4f}  |  F1: {f1_no_emb:.4f}')
print()
print(classification_report(y_te, pred_no_emb, target_names=['Negative', 'Positive']))


### 9b.3 GloVe Frozen vs No Embeddings — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history.history[f'val_{metric}'],        'b-o', ms=4, label='GloVe frozen (model)')
    ax.plot(history_no_emb.history[f'val_{metric}'], 'r-s', ms=4, label='No pretrained emb')
    ax.set_title(f'Validation {title}: GloVe vs Random Init', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(metric.capitalize())
    ax.legend()

plt.tight_layout()
plt.show()


### 9b.4 Optimizer Comparison — Adam vs RMSprop vs SGD+Momentum

Architecture is fixed (CNN with frozen GloVe, dropout=0.5). Only the optimizer changes.

| Optimizer | LR | Notes |
|---|---|---|
| Adam (baseline) | 1e-3 | Already trained as `model` |
| RMSprop | 1e-3 | Adaptive, gradient RMS normalisation |
| SGD + Momentum | 1e-2, mom=0.9 | Classic; may need more epochs |


In [ ]:
from tensorflow.keras.optimizers import RMSprop, SGD

optimizer_configs = [
    ('RMSprop',      RMSprop(learning_rate=1e-3)),
    ('SGD_Momentum', SGD(learning_rate=1e-2, momentum=0.9)),
]

opt_histories = {}
opt_preds     = {}
opt_metrics   = {}

for opt_name, opt_obj in optimizer_configs:
    print(f'\n>>> Training with optimizer: {opt_name}')
    m = Sequential([
        Embedding(vocab_size, embedding_dim,
                  weights=[embedding_matrix],
                  input_length=SEQ_MAX_LEN,
                  trainable=False),
        Conv1D(128, 3, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid'),
    ], name=f'CNN_{opt_name}')

    m.compile(optimizer=opt_obj, loss='binary_crossentropy', metrics=['accuracy'])

    es = EarlyStopping(monitor='val_loss', patience=3,
                       restore_best_weights=True, verbose=0)
    h = m.fit(X_tr, y_tr, epochs=10, batch_size=256,
              validation_data=(X_vl, y_vl), callbacks=[es], verbose=1)

    proba = m.predict(X_te, verbose=0).ravel()
    pred  = (proba > 0.5).astype(int)
    acc   = accuracy_score(y_te, pred)
    f1    = f1_score(y_te, pred)

    opt_histories[opt_name] = h
    opt_preds[opt_name]     = (pred, proba)
    opt_metrics[opt_name]   = {'acc': acc, 'f1': f1}

    print(f'  -> Accuracy: {acc:.4f}  |  F1: {f1:.4f}')


In [ ]:
# Add Adam baseline for comparison
adam_proba = model.predict(X_te, verbose=0).ravel()
adam_pred  = (adam_proba > 0.5).astype(int)
opt_metrics['Adam'] = {
    'acc': accuracy_score(y_te, adam_pred),
    'f1':  f1_score(y_te, adam_pred)
}
opt_preds['Adam'] = (adam_pred, adam_proba)

print('\n=== Optimizer Comparison ===')
print(f'  {"Optimizer":<20} {"Accuracy":>10} {"F1":>10}')
print('  ' + '-'*42)
for name, m in opt_metrics.items():
    print(f'  {name:<20} {m["acc"]:>10.4f} {m["f1"]:>10.4f}')


### 9b.5 Optimizer Comparison — Validation Curves & Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Adam': '#2980b9', 'RMSprop': '#27ae60', 'SGD_Momentum': '#e74c3c'}

# Val accuracy curves
for opt_name, h in [('Adam', history)] + list(opt_histories.items()):
    color = colors.get(opt_name, 'grey')
    axes[0].plot(h.history['val_accuracy'], '-o', ms=4, color=color, label=opt_name)
    axes[1].plot(h.history['val_loss'],     '-o', ms=4, color=color, label=opt_name)

for ax, title, ylabel in zip(axes,
        ['Validation Accuracy by Optimizer', 'Validation Loss by Optimizer'],
        ['Accuracy', 'Loss']):
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend()

plt.tight_layout()
plt.show()

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
names = list(opt_metrics.keys())
accs  = [opt_metrics[n]['acc'] for n in names]
f1s   = [opt_metrics[n]['f1']  for n in names]
x     = np.arange(len(names))
w     = 0.35

bars1 = ax.bar(x - w/2, accs, w, label='Accuracy', color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + w/2, f1s,  w, label='F1 Score',  color='darkorange', alpha=0.85)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0.7, 1.0)
ax.set_title('CNN Optimizer Comparison — Accuracy & F1', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()


### 9b.6 Complete Phase 2 Model Summary

In [ ]:
# Gather metrics for every Phase-2 model in one place
import pandas as pd

p2_summary_rows = []

p2_model_map = [
    ('CNN GloVe frozen, Adam, drop=0.5',    model,       y_test, X_test, None),
    ('CNN GloVe frozen, Adam, drop=0.3',    model2,      y_test, X_test, None),
    ('CNN GloVe trainable, Adam, drop=0.5', model3,      y_test, X_test, None),
    ('CNN No Pretrained Emb, Adam, drop=0.5', model_no_emb, y_te, X_te, None),
]

# Add optimizer variants (evaluated on X_te / y_te)
for opt_name in ['Adam', 'RMSprop', 'SGD_Momentum']:
    pred_o, proba_o = opt_preds[opt_name]
    p2_summary_rows.append({
        'Model': f'CNN GloVe frozen, {opt_name}, drop=0.5',
        'Accuracy': round(accuracy_score(y_te, pred_o), 4),
        'F1':       round(f1_score(y_te, pred_o), 4),
    })

# Evaluate the other models on their respective test sets
for label, m, yt, Xt, _ in p2_model_map:
    proba = m.predict(Xt, verbose=0).ravel()
    pred  = (proba > 0.5).astype(int)
    p2_summary_rows.append({
        'Model':    label,
        'Accuracy': round(accuracy_score(yt, pred), 4),
        'F1':       round(f1_score(yt, pred), 4),
    })

p2_df = (pd.DataFrame(p2_summary_rows)
           .set_index('Model')
           .sort_values('F1', ascending=False))

print(p2_df.to_string())


---
## 10. Evaluation, Plots & Phase 1 vs Phase 2 Comparison

> **Responsibility:** Deep evaluation of all trained models, detailed per-model visualisations,  
> and a final cross-phase comparison between classical ML (Phase 1) and deep learning (Phase 2).

All Phase 1 variables (`best_rf`, `xgb`, `nb_clf`, etc.) and Phase 2 variables  
(`model`, `model2`, `model3`, `autoencoder`) are expected to be in memory from the cells above.


### 10.1  Collect All Model Results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_curve, auc, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13,
                     'axes.labelsize': 11, 'legend.fontsize': 9})

# Phase 1 models
all_models = {}

all_models['KNN'] = {
    'y_true': y_test_knn, 'y_pred': y_pred_knn, 'y_proba': None, 'phase': 1}

all_models['Naive Bayes'] = {
    'y_true': y_test, 'y_pred': y_pred_nb,
    'y_proba': nb_clf.predict_proba(X_test_nb)[:, 1], 'phase': 1}

all_models['Random Forest'] = {
    'y_true': y_test,
    'y_pred': best_rf.predict(X_test_svd_full),
    'y_proba': best_rf.predict_proba(X_test_svd_full)[:, 1], 'phase': 1}

all_models['XGBoost'] = {
    'y_true': y_test,
    'y_pred': xgb.predict(X_test_svd_full),
    'y_proba': xgb.predict_proba(X_test_svd_full)[:, 1], 'phase': 1}

# Phase 2 models (all evaluated on X_te / y_te)
def get_dl(m, X, y, proba_in=None):
    proba = proba_in if proba_in is not None else m.predict(X, verbose=0).ravel()
    pred  = (proba > 0.5).astype(int)
    return {'y_true': y, 'y_pred': pred, 'y_proba': proba, 'phase': 2}

all_models['CNN GloVe frozen, Adam, drop=0.5']    = get_dl(model,       X_te, y_te)
all_models['CNN GloVe frozen, Adam, drop=0.3']    = get_dl(model2,      X_te, y_te)
all_models['CNN GloVe trainable, Adam, drop=0.5'] = get_dl(model3,      X_te, y_te)
all_models['CNN No Pretrained Emb (random init)'] = get_dl(model_no_emb, X_te, y_te)

# Optimizer variants (predictions already stored)
for opt_name in ['Adam', 'RMSprop', 'SGD_Momentum']:
    pred_o, proba_o = opt_preds[opt_name]
    all_models[f'CNN GloVe frozen, {opt_name}'] = {
        'y_true': y_te, 'y_pred': pred_o, 'y_proba': proba_o, 'phase': 2}

print(f'Loaded {sum(1 for v in all_models.values() if v["phase"]==1)} Phase-1 models '
      f'and {sum(1 for v in all_models.values() if v["phase"]==2)} Phase-2 models.')


### 10.2  Full Metrics Table

In [ ]:
rows = []
for name, d in all_models.items():
    yt, yp = d['y_true'], d['y_pred']
    rows.append({
        'Model':     name,
        'Phase':     d['phase'],
        'Accuracy':  round(accuracy_score(yt, yp), 4),
        'Precision': round(precision_score(yt, yp, zero_division=0), 4),
        'Recall':    round(recall_score(yt, yp, zero_division=0), 4),
        'F1':        round(f1_score(yt, yp, zero_division=0), 4),
    })

metrics_df = pd.DataFrame(rows).set_index('Model').sort_values('F1', ascending=False)
print(metrics_df.to_string())


### 10.3  Confusion Matrices — All Models

In [ ]:
n_models = len(all_models)
n_cols   = 4
n_rows   = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
axes_flat = axes.flatten()

for ax, (name, d) in zip(axes_flat, all_models.items()):
    cm = confusion_matrix(d['y_true'], d['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Phase {d["phase"]} — {name}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

# Hide unused subplots
for ax in axes_flat[n_models:]:
    ax.set_visible(False)

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


### 10.4  Per-Class Classification Reports

In [ ]:
for name, d in all_models.items():
    print(f'\n{"─"*60}')
    print(f'  Phase {d["phase"]} | {name}')
    print(f'{"─"*60}')
    print(classification_report(d['y_true'], d['y_pred'],
                                 target_names=['Negative','Positive'],
                                 zero_division=0))


### 10.5  ROC Curves — Phase 1 vs Phase 2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_p1 = ['#e74c3c', '#e67e22', '#8e44ad', '#16a085']
colors_p2 = ['#2980b9', '#27ae60', '#d35400']

for ax, (phase, color_list, title) in zip(
        axes,
        [(1, colors_p1, 'Phase 1 — Classical ML'),
         (2, colors_p2, 'Phase 2 — Deep Learning')]):

    phase_models = {k: v for k, v in all_models.items() if v['phase'] == phase}
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)

    for (name, d), color in zip(phase_models.items(), color_list):
        if d['y_proba'] is None:
            continue
        fpr, tpr, _ = roc_curve(d['y_true'], d['y_proba'])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{name}  (AUC={roc_auc:.3f})')

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])

plt.suptitle('ROC Curves — Phase 1 vs Phase 2', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 10.6  Accuracy & F1 Bar Chart — All Models

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

labels  = metrics_df.index.tolist()
accs    = metrics_df['Accuracy'].values
f1s     = metrics_df['F1'].values
phases  = metrics_df['Phase'].values

x     = np.arange(len(labels))
width = 0.35

# Colour by phase
bar_colors = ['#e74c3c' if p == 1 else '#2980b9' for p in phases]

bars_acc = ax.bar(x - width/2, accs, width, color=bar_colors, alpha=0.85, label='Accuracy')
bars_f1  = ax.bar(x + width/2, f1s,  width, color=bar_colors, alpha=0.55, label='F1 Score', hatch='//')

# Annotate bars
for bar in bars_acc:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7.5)
for bar in bars_f1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.set_ylim(0.6, 1.02)
ax.set_ylabel('Score')
ax.set_title('All Models — Accuracy & F1 (red = Phase 1, blue = Phase 2)', fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='Phase 1 — Classical ML'),
    Patch(facecolor='#2980b9', label='Phase 2 — Deep Learning'),
    Patch(facecolor='grey', alpha=0.85, label='Accuracy'),
    Patch(facecolor='grey', alpha=0.55, hatch='//', label='F1 Score'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()


### 10.7  Radar Chart — Best Model per Phase

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.path import Path as MPath

# Pick best Phase-1 and best Phase-2 by F1
best_p1_name = metrics_df[metrics_df['Phase'] == 1]['F1'].idxmax()
best_p2_name = metrics_df[metrics_df['Phase'] == 2]['F1'].idxmax()

metric_keys = ['Accuracy', 'Precision', 'Recall', 'F1']
vals_p1 = metrics_df.loc[best_p1_name, metric_keys].values.astype(float)
vals_p2 = metrics_df.loc[best_p2_name, metric_keys].values.astype(float)

N = len(metric_keys)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close polygon

def radar_row(vals):
    v = vals.tolist()
    v += v[:1]
    return v

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

ax.plot(angles, radar_row(vals_p1), 'o-', linewidth=2, color='#e74c3c', label=f'P1: {best_p1_name}')
ax.fill(angles, radar_row(vals_p1), alpha=0.15, color='#e74c3c')

ax.plot(angles, radar_row(vals_p2), 's-', linewidth=2, color='#2980b9', label=f'P2: {best_p2_name}')
ax.fill(angles, radar_row(vals_p2), alpha=0.15, color='#2980b9')

ax.set_thetagrids(np.degrees(angles[:-1]), metric_keys, fontsize=11)
ax.set_ylim(0.6, 1.0)
ax.set_title('Best Phase-1 vs Best Phase-2 Model', fontweight='bold', pad=15)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.show()


### 10.8  Phase 1 vs Phase 2 — Qualitative Comparison Table

In [ ]:
comparison_data = {
    'Criterion': [
        'Best Accuracy', 'Best F1',
        'Training Time', 'Inference Speed',
        'Interpretability', 'Requires GPU',
        'Handles OOV Words', 'Scales to More Data',
    ],
    'Phase 1 — Classical ML': [
        f'{metrics_df[metrics_df["Phase"]==1]["Accuracy"].max():.4f}',
        f'{metrics_df[metrics_df["Phase"]==1]["F1"].max():.4f}',
        'Minutes (CPU)', 'Very fast',
        'High (feature weights)', 'No',
        'No (fixed vocab)', 'Limited (memory)',
    ],
    'Phase 2 — Deep Learning': [
        f'{metrics_df[metrics_df["Phase"]==2]["Accuracy"].max():.4f}',
        f'{metrics_df[metrics_df["Phase"]==2]["F1"].max():.4f}',
        'Minutes–Hours (GPU)', 'Fast (batched)',
        'Low (black-box)', 'Recommended',
        'Yes (embeddings)', 'Excellent',
    ],
}

comp_df = pd.DataFrame(comparison_data).set_index('Criterion')
print(comp_df.to_string())


### 10.9  CNN Training Curves — Epoch-by-Epoch Analysis

In [ ]:
# history is the CNN baseline history object from Person 2's section
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

epochs_range = range(1, len(history.history['accuracy']) + 1)

axes[0].plot(epochs_range, history.history['accuracy'],     'b-o', label='Train Acc', ms=4)
axes[0].plot(epochs_range, history.history['val_accuracy'], 'r-s', label='Val Acc',   ms=4)
axes[0].set_title('CNN Baseline — Accuracy per Epoch', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(epochs_range, history.history['loss'],     'b-o', label='Train Loss', ms=4)
axes[1].plot(epochs_range, history.history['val_loss'], 'r-s', label='Val Loss',   ms=4)
axes[1].set_title('CNN Baseline — Loss per Epoch', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Binary Cross-Entropy')
axes[1].legend()

# Highlight best epoch
best_epoch = np.argmin(history.history['val_loss']) + 1
for ax in axes:
    ax.axvline(best_epoch, color='green', linestyle='--', linewidth=1.2,
               label=f'Best epoch ({best_epoch})')
    ax.legend()

plt.tight_layout()
plt.show()
print(f'Early stopping triggered at epoch {len(epochs_range)} — best val_loss at epoch {best_epoch}')


### 10.10  Final Leaderboard — All Models Ranked by F1

In [ ]:
print('=' * 70)
print(f'  {"Rank":<5} {"Model":<38} {"Phase":<7} {"Acc":>8} {"F1":>8}')
print('=' * 70)

for rank, (name, row) in enumerate(metrics_df.iterrows(), 1):
    phase_tag = 'DL' if row['Phase'] == 2 else 'ML'
    marker    = ' ◀ BEST' if rank == 1 else ''
    print(f'  {rank:<5} {name:<38} {phase_tag:<7} {row["Accuracy"]:>8.4f} {row["F1"]:>8.4f}{marker}')

print('=' * 70)
best_overall = metrics_df.index[0]
best_phase   = 'Deep Learning (Phase 2)' if metrics_df.iloc[0]['Phase'] == 2 else 'Classical ML (Phase 1)'
print(f'\n  Best overall model : {best_overall}')
print(f'  Phase              : {best_phase}')
print(f'  Accuracy           : {metrics_df.iloc[0]["Accuracy"]:.4f}')
print(f'  F1 Score           : {metrics_df.iloc[0]["F1"]:.4f}')


---
## 11. GRU Model

A **Gated Recurrent Unit (GRU)** is a recurrent neural network that captures sequential dependencies in text — it reads the tweet token-by-token, maintaining a hidden state that acts as memory. Unlike the CNN (which detects local n-gram patterns via convolution), the GRU models the *temporal/positional* flow of words, making it well-suited for sentiment tasks where word order matters.

### 11.1 Build & Train the GRU Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# --- Architecture ---
gru_model = Sequential([
    # Pre-trained GloVe embeddings (frozen — we let the GRU learn on top)
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=SEQ_MAX_LEN,
        trainable=False
    ),

    # GRU layer returns only the final hidden state (return_sequences=False)
    GRU(128, dropout=0.2, recurrent_dropout=0.2),

    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')   # binary output
], name='GRU_GloVe_frozen')

gru_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
gru_model.summary()


In [ ]:
es_gru = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

history_gru = gru_model.fit(
    X_tr, y_tr,
    epochs=10,
    batch_size=256,
    validation_data=(X_vl, y_vl),
    callbacks=[es_gru],
    verbose=1
)


### 11.2 Evaluate on the Test Set

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

gru_proba = gru_model.predict(X_te, verbose=0).ravel()
gru_pred  = (gru_proba > 0.5).astype(int)

gru_acc = accuracy_score(y_te, gru_pred)
gru_f1  = f1_score(y_te, gru_pred)

print(f'GRU Accuracy : {gru_acc:.4f}')
print(f'GRU F1 Score : {gru_f1:.4f}')
print()
print(classification_report(y_te, gru_pred, target_names=['Negative', 'Positive']))


In [ ]:
# Confusion Matrix
cm_gru = confusion_matrix(y_te, gru_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_gru, annot=True, fmt='d', cmap='Purples',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive'],
    ax=ax
)
ax.set_title('GRU — Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('plot_gru_confusion_matrix.png', bbox_inches='tight')
plt.show()


### 11.3 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history_gru.history[metric],          'b-o', ms=4, label='Train')
    ax.plot(history_gru.history[f'val_{metric}'], 'r-s', ms=4, label='Validation')
    best_ep = (int.__sub__(len(history_gru.history[metric]), 1)
               if metric == 'accuracy'
               else history_gru.history[f'val_{metric}'].index(
                   min(history_gru.history[f'val_{metric}'])))
    ax.axvline(best_ep, color='green', linestyle='--', linewidth=1.2, label=f'Best epoch ({best_ep+1})')
    ax.set_title(f'GRU — {title} per Epoch', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend()

plt.tight_layout()
plt.savefig('plot_gru_training_curves.png', bbox_inches='tight')
plt.show()


### 11.4 GRU vs CNN — Head-to-Head Comparison

In [ ]:
import numpy as np

# Collect metrics for CNN baseline (model, evaluated on X_te/y_te) and GRU
cnn_proba_te = model.predict(X_te, verbose=0).ravel()
cnn_pred_te  = (cnn_proba_te > 0.5).astype(int)

comparison = {
    'CNN (GloVe frozen, Adam, drop=0.5)': {
        'Accuracy': accuracy_score(y_te, cnn_pred_te),
        'F1':       f1_score(y_te, cnn_pred_te),
    },
    'GRU (GloVe frozen, Adam)': {
        'Accuracy': gru_acc,
        'F1':       gru_f1,
    },
}

import pandas as pd
comp_df = pd.DataFrame(comparison).T
print(comp_df.to_string(float_format='{:.4f}'.format))

# Side-by-side bar chart
metrics_list = ['Accuracy', 'F1']
x     = np.arange(len(metrics_list))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars_cnn = ax.bar(x - width/2, comp_df.loc['CNN (GloVe frozen, Adam, drop=0.5)', metrics_list],
                  width, label='CNN', color='#2980b9', alpha=0.85)
bars_gru = ax.bar(x + width/2, comp_df.loc['GRU (GloVe frozen, Adam)', metrics_list],
                  width, label='GRU', color='#8e44ad', alpha=0.85)

for bar in list(bars_cnn) + list(bars_gru):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(metrics_list, fontsize=12)
ax.set_ylim(0.7, 1.0)
ax.set_ylabel('Score')
ax.set_title('CNN vs GRU — Performance Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('plot_gru_vs_cnn.png', bbox_inches='tight')
plt.show()


### 11.5 Add GRU to the All-Models Leaderboard

Register the GRU in `all_models` so it appears in Section 10's final leaderboard and ROC chart.

In [ ]:
# Register GRU in the global model registry
all_models['GRU (GloVe frozen, Adam)'] = {
    'y_true':  y_te,
    'y_pred':  gru_pred,
    'y_proba': gru_proba,
    'phase':   2
}

# Recompute the metrics table (same code as Section 10.2)
rows = []
for name, d in all_models.items():
    from sklearn.metrics import precision_score, recall_score
    yt, yp = d['y_true'], d['y_pred']
    rows.append({
        'Model':     name,
        'Phase':     d['phase'],
        'Accuracy':  round(accuracy_score(yt, yp), 4),
        'Precision': round(precision_score(yt, yp, zero_division=0), 4),
        'Recall':    round(recall_score(yt, yp, zero_division=0), 4),
        'F1':        round(f1_score(yt, yp, zero_division=0), 4),
    })

metrics_df = pd.DataFrame(rows).set_index('Model').sort_values('F1', ascending=False)
print('Updated leaderboard (top 10):')
print(metrics_df.head(10).to_string())


## 12.LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout
)

lstm_model = Sequential([

    # Embedding Layer
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=SEQ_MAX_LEN,
        trainable=True
    ),

    # LSTM Layer
    LSTM(
        64,
        dropout=0.2,
        return_sequences=False
    ),

    # Regularization
    Dropout(0.4),

    # Dense Layer
    Dense(64, activation='relu'),

    Dropout(0.3),

    # Output Layer
    Dense(1, activation='sigmoid')
])


lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


lstm_model.summary()

In [ ]:
from sklearn.model_selection import train_test_split

# =========================================
# Train / Validation / Test Split
# =========================================

# 60% Train
# 20% Validation
# 20% Test

X_train, X_temp, y_train, y_temp = train_test_split(
    X_token_padded,
    Y,
    test_size=0.40,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

# =========================================
# Shapes
# =========================================

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau 

reduce_lr = ReduceLROnPlateau( 
    monitor='val_loss', 
    factor=0.5, 
    patience=1 
)

early_stop = EarlyStopping(
    patience=3,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predict probabilities
pred_lstm = lstm_model.predict(X_test)

# Convert to binary labels
pred_lstm = (pred_lstm > 0.5).astype(int)

# Metrics
acc_lstm = accuracy_score(y_test, pred_lstm)
f1_lstm = f1_score(y_test, pred_lstm)

print(f"LSTM Accuracy: {acc_lstm:.4f}")
print(f"LSTM F1 Score: {f1_lstm:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_lstm))

print("\nClassification Report:")
print(classification_report(y_test, pred_lstm))

In [ ]:
import matplotlib.pyplot as plt

# Accuracy
plt.plot(history_lstm.history['accuracy'])
plt.plot(history_lstm.history['val_accuracy'])

plt.title('LSTM Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.legend(['Train', 'Validation'])

plt.show()

# Loss
plt.plot(history_lstm.history['loss'])
plt.plot(history_lstm.history['val_loss'])

plt.title('LSTM Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.legend(['Train', 'Validation'])

plt.show()

---
## 13. Vanilla RNN (`SimpleRNN`)

A **vanilla recurrent neural network** uses the classic Elman-style update: each timestep combines the current input with the previous hidden state through a single learned weight matrix and a nonlinearity (typically `tanh`). Keras exposes this as `SimpleRNN`. It is the simplest sequential model (unlike **GRU** or **LSTM**, it has no separate gates for forgetting or updating), so it often needs careful regularisation and can be harder to train on long sequences — but it is the standard baseline for demonstrating **pure RNN** behaviour on text.

The model below reuses the same **frozen GloVe** embedding matrix, padded sequences (`SEQ_MAX_LEN`), and train/validation/test splits as the CNN and GRU sections, so results are directly comparable.

### 13.1 Build the RNN classifier

We stack **Embedding → SimpleRNN → Dropout → Dense → sigmoid**, matching the GRU depth (two dense blocks after the recurrent layer) so differences come mainly from the recurrent cell type.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam

rnn_model = Sequential(
    [
        Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            input_length=SEQ_MAX_LEN,
            trainable=False,
        ),
        SimpleRNN(128, dropout=0.2, recurrent_dropout=0.2),
        Dropout(0.5),
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid"),
    ],
    name="SimpleRNN_GloVe_frozen",
)

rnn_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

rnn_model.summary()

### 13.2 Train with early stopping

Same training budget as the GRU (`epochs=10`, `batch_size=256`, `validation_data=(X_vl, y_vl)`) so training time and convergence are comparable.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Needs X_tr, y_tr, X_vl, y_vl from the "9b.1 Shared Train / Val / Test Split" cell
# (the same cell that sets Y_dl from X_token_padded).
if "X_tr" not in globals() or "y_tr" not in globals():
    raise RuntimeError(
        "Run the split cell first: it does train_test_split on X_token_padded and Y_dl "
        "and defines X_tr, y_tr, X_vl, y_vl, X_te, y_te."
    )

es_rnn = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1,
)

history_rnn = rnn_model.fit(
    X_tr,
    y_tr,
    epochs=100,
    batch_size=256,
    validation_data=(X_vl, y_vl),
    callbacks=[es_rnn],
    verbose=1,
)

### 13.3 Evaluate on the test set

Binary predictions use a **0.5** threshold on the sigmoid output (same convention as the CNN and GRU sections).

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

rnn_proba = rnn_model.predict(X_te, verbose=0).ravel()
rnn_pred = (rnn_proba > 0.5).astype(int)

rnn_acc = accuracy_score(y_te, rnn_pred)
rnn_f1 = f1_score(y_te, rnn_pred)

print(f"SimpleRNN Accuracy : {rnn_acc:.4f}")
print(f"SimpleRNN F1 Score : {rnn_f1:.4f}")
print()
print(classification_report(y_te, rnn_pred, target_names=["Negative", "Positive"]))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm_rnn = confusion_matrix(y_te, rnn_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_rnn,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
    ax=ax,
)
ax.set_title("SimpleRNN — Confusion Matrix", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()
plt.savefig("plot_simple_rnn_confusion_matrix.png", bbox_inches="tight")
plt.show()

### 13.4 Training curves

Training and validation **accuracy** and **loss** by epoch; the vertical line marks the epoch chosen by early stopping (best `val_loss` when `restore_best_weights=True`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs = np.arange(1, len(history_rnn.history["accuracy"]) + 1)
best_epoch = int(np.argmin(history_rnn.history["val_loss"])) + 1

for ax, metric, title in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
    ax.plot(epochs, history_rnn.history[metric], "b-o", ms=4, label="Train")
    ax.plot(epochs, history_rnn.history[f"val_{metric}"], "r-s", ms=4, label="Validation")
    ax.axvline(
        best_epoch,
        color="green",
        linestyle="--",
        linewidth=1.2,
        label=f"Best val_loss (epoch {best_epoch})",
    )
    ax.set_title(f"SimpleRNN — {title} per Epoch", fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend()

plt.tight_layout()
plt.savefig("plot_simple_rnn_training_curves.png", bbox_inches="tight")
plt.show()

print(
    f"Training ran {len(epochs)} epoch(s); best val_loss at epoch {best_epoch} "
    f"(weights restored via EarlyStopping(..., restore_best_weights=True))."
)